# 02 — Add Match Metadata and Chronological Order
Load match metadata, merge it into extracted shot events, derive clean chronological fields,
and save chronologically ordered datasets for later feature engineering.

In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

import pandas as pd
from IPython.display import display
from src.config import INTERIM_DIR as DATA_DIR, MATCH_DIR, ALL_LEAGUES, TRAIN_LEAGUES, TEST_LEAGUES
from src.data_loader import load_parquet, load_matches
from src.preprocessing import (
    build_all_match_df, merge_match_metadata,
    add_chronological_fields, split_train_test,
)

## Step 0: Path validation

In [2]:
# Step 0: Path validation
assert (DATA_DIR / "wyscout_shots.parquet").exists(),       "Run 01_extract_shots first"
assert (DATA_DIR / "wyscout_train_shots.parquet").exists(), "Run 01_extract_shots first"
assert (DATA_DIR / "wyscout_test_shots.parquet").exists(),  "Run 01_extract_shots first"
assert MATCH_DIR.exists(),                                   f"Missing: {MATCH_DIR}"
assert (MATCH_DIR / "matches_England.json").exists(),       "England match file not found"
print("Paths OK")

Paths OK


## Step 1: Load shot tables from notebook 01

In [3]:
# Step 1: Load shot tables from notebook 01
shots_orig  = load_parquet(DATA_DIR / "wyscout_shots.parquet")
train_orig  = load_parquet(DATA_DIR / "wyscout_train_shots.parquet")
test_orig   = load_parquet(DATA_DIR / "wyscout_test_shots.parquet")

assert len(shots_orig) > 0
assert len(shots_orig) == len(train_orig) + len(test_orig), "Combined ≠ train + test"

EXPECTED_SHOT_COLS = {
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName",
    "x", "y", "is_goal", "is_penalty", "is_direct_free_kick",
}
assert EXPECTED_SHOT_COLS.issubset(shots_orig.columns), \
    f"Missing columns: {EXPECTED_SHOT_COLS - set(shots_orig.columns)}"

# Normalise flag dtypes immediately after load (parquet may not preserve int8)
for df in [shots_orig, train_orig, test_orig]:
    df["is_penalty"]          = df["is_penalty"].astype("int8")
    df["is_direct_free_kick"] = df["is_direct_free_kick"].astype("int8")

print(f"Combined: {len(shots_orig):,}  |  Train: {len(train_orig):,}  |  Test: {len(test_orig):,}")

Combined: 43,040  |  Train: 34,159  |  Test: 8,881


## Step 2: Helper functions

In [4]:
# Helpers now live in src.preprocessing

## Step 3: Load and flatten match metadata per league

In [5]:
# Step 3: Load and flatten match metadata per league
# verbose=True preserves the current per-league console output exactly.
matches_df = build_all_match_df(ALL_LEAGUES, load_matches, verbose=True)

England: 380 matches
France: 380 matches
Germany: 306 matches
Italy: 380 matches
Spain: 380 matches

Total matches: 1,826


## Step 4: Merge match metadata into shots

In [6]:
# Step 4: Merge match metadata into shots
shots = merge_match_metadata(shots_orig, matches_df)
print(f"Merged shots: {len(shots):,}")
shots[["league", "matchId", "match_datetime_utc", "gameweek", "match_label"]].head()

Merged shots: 43,040


,league,matchId,match_datetime_utc,gameweek,match_label
0,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
1,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
2,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
3,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"
4,France,2500686,2017-08-06 15:00:00+00:00,1,"Angers SCO - Bordeaux, 2 - 2"


## Step 5: Build chronological ordering fields

> `seconds_from_match_start` is a sort key only — it is NOT true ball-in-play time.  
> Wyscout's `eventSec` resets at the start of each period.

In [7]:
# Step 5: Build chronological ordering fields + rebuild train/test splits
shots = add_chronological_fields(shots)
train_shots, test_shots = split_train_test(shots)
print(f"Train: {len(train_shots):,}  |  Test: {len(test_shots):,}")

Train: 34,159  |  Test: 8,881


## Step 6: Sanity checks

In [8]:
ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]
assert not shots[ID_COLS].duplicated().any(), "Duplicate shot identities after merge"

for col in ["match_datetime_utc", "period_sort_key", "seconds_from_match_start"]:
    assert shots[col].notna().all(), f"Null values in {col}"

assert shots["seconds_from_match_start"].ge(0).all(), "Negative seconds_from_match_start"
assert shots["period_sort_key"].isin([1,2,3,4,5]).all(), "Unexpected period_sort_key values"

# Attempt-type flag checks
assert shots["is_penalty"].isin([0, 1]).all(), "is_penalty not binary"
assert shots["is_direct_free_kick"].isin([0, 1]).all(), "is_direct_free_kick not binary"
assert not ((shots["is_penalty"] == 1) & (shots["is_direct_free_kick"] == 1)).any(), \
    "A row has both is_penalty and is_direct_free_kick = 1"

# Event/subevent pair consistency
assert (shots.loc[shots["is_penalty"] == 1, "eventName"] == "Free Kick").all(), \
    "Penalty row has unexpected eventName"
assert (shots.loc[shots["is_penalty"] == 1, "subEventName"] == "Penalty").all(), \
    "Penalty row has unexpected subEventName"
assert (shots.loc[shots["is_direct_free_kick"] == 1, "eventName"] == "Free Kick").all(), \
    "Direct FK shot row has unexpected eventName"
assert (shots.loc[shots["is_direct_free_kick"] == 1, "subEventName"] == "Free kick shot").all(), \
    "Direct FK shot row has unexpected subEventName"
assert (shots.loc[
    (shots["is_penalty"] == 0) & (shots["is_direct_free_kick"] == 0), "eventName"
] == "Shot").all(), "Standard Shot/Shot rows have unexpected eventName"

print("All merge-quality checks passed")
print(shots[["match_datetime_utc","gameweek","period_sort_key","seconds_from_match_start"]].describe())

All merge-quality checks passed
           gameweek  period_sort_key  seconds_from_match_start
count  43040.000000     43040.000000              43040.000000
mean      19.011222         1.537384               2928.937374
std       10.855843         0.498606               1589.921143
min        1.000000         1.000000                  1.238426
25%       10.000000         1.000000               1587.743672
50%       19.000000         2.000000               2940.989250
75%       28.000000         2.000000               4296.088754
max       38.000000         2.000000               6190.826794


In [9]:
assert shots["match_datetime_utc"].is_monotonic_increasing, "Shots not sorted by match_datetime_utc"

# Within each match, period_sort_key must be non-decreasing
within_match_period_ok = (
    shots.groupby(["league","matchId"])["period_sort_key"]
         .apply(lambda s: s.diff().fillna(0).ge(0).all())
         .all()
)
assert within_match_period_ok, "period_sort_key not non-decreasing within a match"

# shot_sequence_in_match must run 1..n with no gaps and no duplicates
seq = shots.groupby(["league","matchId"])["shot_sequence_in_match"].agg(["min","max","count"])
assert (seq["min"] == 1).all(), "shot_sequence_in_match does not start at 1 for every match"
assert (seq["max"] == seq["count"]).all(), "Gaps or duplicates in shot_sequence_in_match"

print(f"Earliest match: {shots['match_datetime_utc'].min()}")
print(f"Latest match:   {shots['match_datetime_utc'].max()}")
print(f"Matches in shots: {shots[['league','matchId']].drop_duplicates().shape[0]:,}")

Earliest match: 2017-08-04 18:45:00+00:00
Latest match:   2018-05-20 18:45:00+00:00
Matches in shots: 1,826


In [10]:
summary = shots.groupby("league").agg(
    n_matches=("matchId", "nunique"),
    n_shots=("matchId", "count"),
    min_date=("match_date", "min"),
    max_date=("match_date", "max"),
    min_gameweek=("gameweek", "min"),
    max_gameweek=("gameweek", "max"),
)
display(summary)

,n_matches,n_shots,min_date,max_date,min_gameweek,max_gameweek
league,,,,,,
England,380,8881,2017-08-11,2018-05-13,1,38
France,380,8977,2017-08-04,2018-05-19,1,38
Germany,306,7290,2017-08-18,2018-05-12,1,34
Italy,380,9347,2017-08-19,2018-05-20,1,38
Spain,380,8545,2017-08-18,2018-05-20,1,38


## Step 7: Save outputs

In [11]:
SAVE_COLS = [
    # Shot identity / event fields
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "eventName", "subEventName",
    "x", "y", "is_goal", "is_penalty", "is_direct_free_kick",
    # Match metadata (scalar only)
    "match_datetime_utc", "match_date", "match_date_local_raw",
    "gameweek", "roundId", "seasonId", "competitionId",
    "match_label", "match_status", "match_duration", "winner",
    # Chronological ordering fields
    "period_sort_key", "seconds_from_match_start",
    "shot_sequence_in_match", "shot_sequence_team_in_match",
]

shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_shots_with_matches.parquet", index=False)
train_shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_train_shots_with_matches.parquet", index=False)
test_shots[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_test_shots_with_matches.parquet", index=False)
shots[SAVE_COLS].head(100).to_csv(DATA_DIR / "wyscout_shots_with_matches_sample.csv", index=False)

print(f"Saved {len(shots):,} combined  → wyscout_shots_with_matches.parquet")
print(f"Saved {len(train_shots):,} train    → wyscout_train_shots_with_matches.parquet")
print(f"Saved {len(test_shots):,} test     → wyscout_test_shots_with_matches.parquet")
print("Saved 100-row sample        → wyscout_shots_with_matches_sample.csv")

Saved 43,040 combined  → wyscout_shots_with_matches.parquet
Saved 34,159 train    → wyscout_train_shots_with_matches.parquet
Saved 8,881 test     → wyscout_test_shots_with_matches.parquet
Saved 100-row sample        → wyscout_shots_with_matches_sample.csv


In [12]:
check = pd.read_parquet(DATA_DIR / "wyscout_shots_with_matches.parquet")
assert check.shape == shots[SAVE_COLS].shape, "Shape mismatch on reload"
assert set(SAVE_COLS) == set(check.columns), "Column mismatch on reload"
assert pd.api.types.is_datetime64_any_dtype(check["match_datetime_utc"]), \
    "match_datetime_utc not datetime on reload"
assert check["is_penalty"].isin([0, 1]).all(), "is_penalty not binary on reload"
assert check["is_direct_free_kick"].isin([0, 1]).all(), "is_direct_free_kick not binary on reload"
print(check.shape)
check.head()

(43040, 28)


,league,matchId,playerId,teamId,eventSec,matchPeriod,eventName,subEventName,x,y,...,seasonId,competitionId,match_label,match_status,match_duration,winner,period_sort_key,seconds_from_match_start,shot_sequence_in_match,shot_sequence_team_in_match
0,France,2500691,353833,19830,241.432822,1H,Shot,Shot,91,30,...,181189,412,"Monaco - Toulouse, 3 - 2",Played,Regular,19830,1,241.432822,1,1
1,France,2500691,288423,3780,339.735142,1H,Shot,Shot,92,59,...,181189,412,"Monaco - Toulouse, 3 - 2",Played,Regular,19830,1,339.735142,2,1
2,France,2500691,3450,19830,576.442938,1H,Shot,Shot,91,54,...,181189,412,"Monaco - Toulouse, 3 - 2",Played,Regular,19830,1,576.442938,3,2
3,France,2500691,28922,19830,742.946020,1H,Shot,Shot,82,51,...,181189,412,"Monaco - Toulouse, 3 - 2",Played,Regular,19830,1,742.946020,4,3
4,France,2500691,353833,19830,828.687717,1H,Shot,Shot,91,69,...,181189,412,"Monaco - Toulouse, 3 - 2",Played,Regular,19830,1,828.687717,5,4
